# 🎭 Deepfake Detection System
### Spatial-Temporal CNN + LSTM Pipeline

**Team Members:** [Your Name] | [Friend's Name]  
**Dataset:** FaceForensics++ (Kaggle)  
**Model:** EfficientNet-B0 (CNN) + LSTM  

---
**Phases:**
1. Setup & Mount Drive
2. Install Dependencies
3. Dataset Preparation
4. Frame Extraction
5. CNN Feature Extractor (EfficientNet)
6. LSTM Temporal Classifier
7. Training
8. Evaluation
9. Gradio Demo

## 📌 Phase 1 — Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

# ✅ Set your project paths here
PROJECT_PATH = '/content/drive/MyDrive/deepfake-project'
DATASET_PATH = f'{PROJECT_PATH}/dataset'
MODELS_PATH  = f'{PROJECT_PATH}/models'
FRAMES_PATH  = f'{PROJECT_PATH}/frames'

import os
for path in [DATASET_PATH, MODELS_PATH, FRAMES_PATH,
             f'{FRAMES_PATH}/real', f'{FRAMES_PATH}/fake']:
    os.makedirs(path, exist_ok=True)

print('✅ Drive mounted and folders ready!')

Mounted at /content/drive
✅ Drive mounted and folders ready!


## 📌 Phase 2 — Install Dependencies

In [2]:
!pip install -q timm opencv-python-headless gradio kaggle
print('✅ All packages installed!')

✅ All packages installed!


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import timm
import cv2
import numpy as np
import os
import glob
from PIL import Image
from sklearn.metrics import roc_auc_score, accuracy_score
from tqdm import tqdm
import matplotlib.pyplot as plt

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ Using device: {DEVICE}')

✅ Using device: cuda


## 📌 Phase 3 — Download Dataset from Kaggle

In [ ]:
# Upload your kaggle.json API key first:
# Kaggle → Account → API → Create New Token → upload the file below

from google.colab import files
print('Upload your kaggle.json file:')
uploaded = files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print('✅ Kaggle API configured!')

In [ ]:
import os

# ⚠️ Update your NEW Kaggle token here after regenerating
os.environ['KAGGLE_USERNAME'] = 'shashankkodali6'
os.environ['KAGGLE_KEY'] = 'KGAT_abf32b2311e91aee6bdd87eaaf171d6b'

# Configure kaggle
!mkdir -p ~/.kaggle
!echo '{"username":"shashankkodali6","key":"KGAT_abf32b2311e91aee6bdd87eaaf171d6b"}' > ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

print('✅ Kaggle configured!')

# Download dataset
os.chdir(DATASET_PATH)
!kaggle datasets download -d hungle3401/faceforensics --unzip

print('✅ Dataset downloaded!')
print(os.listdir(DATASET_PATH))

✅ Kaggle configured!
Dataset URL: https://www.kaggle.com/datasets/hungle3401/faceforensics
License(s): DbCL-1.0
 54% 1.47G/2.73G [01:40<01:25, 15.9MB/s]

## 📌 Phase 4 — Frame Extraction from Videos

In [ ]:
def extract_frames(video_path, output_dir, num_frames=10):
    """Extract evenly spaced frames from a video."""
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total == 0:
        cap.release()
        return 0

    indices = np.linspace(0, total - 1, num_frames, dtype=int)
    saved = 0
    frame_paths = []

    for i, idx in enumerate(indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame_resized = cv2.resize(frame_rgb, (224, 224))
            save_path = os.path.join(output_dir, f'frame_{i:03d}.jpg')
            Image.fromarray(frame_resized).save(save_path)
            frame_paths.append(save_path)
            saved += 1

    cap.release()
    return saved


def process_video_folder(video_folder, frames_output, label, max_videos=500):
    """Process all videos in a folder."""
    video_files = glob.glob(os.path.join(video_folder, '*.mp4'))[:max_videos]
    print(f'Processing {len(video_files)} {label} videos...')

    for vf in tqdm(video_files):
        video_name = os.path.splitext(os.path.basename(vf))[0]
        out_dir = os.path.join(frames_output, label, video_name)
        os.makedirs(out_dir, exist_ok=True)
        extract_frames(vf, out_dir, num_frames=10)

    print(f'✅ Done extracting {label} frames!')


# ⚠️ Update these paths to match your downloaded dataset structure
REAL_VIDEOS = os.path.join(DATASET_PATH, 'original_sequences')  # adjust if needed
FAKE_VIDEOS = os.path.join(DATASET_PATH, 'manipulated_sequences')  # adjust if needed

process_video_folder(REAL_VIDEOS, FRAMES_PATH, 'real', max_videos=500)
process_video_folder(FAKE_VIDEOS, FRAMES_PATH, 'fake', max_videos=500)

## 📌 Phase 5 — Dataset Class

In [ ]:
class DeepfakeDataset(Dataset):
    """
    Returns a sequence of frames per video.
    Shape: (num_frames, C, H, W)
    Label: 0 = real, 1 = fake
    """
    def __init__(self, frames_root, num_frames=10, transform=None):
        self.samples = []  # list of (video_frame_dir, label)
        self.num_frames = num_frames
        self.transform = transform

        for label_name, label_idx in [('real', 0), ('fake', 1)]:
            label_dir = os.path.join(frames_root, label_name)
            if not os.path.exists(label_dir):
                continue
            for video_dir in os.listdir(label_dir):
                full_path = os.path.join(label_dir, video_dir)
                if os.path.isdir(full_path):
                    self.samples.append((full_path, label_idx))

        print(f'Dataset: {len(self.samples)} videos found.')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        video_dir, label = self.samples[idx]
        frame_files = sorted(glob.glob(os.path.join(video_dir, '*.jpg')))

        # Pad or trim to exactly num_frames
        if len(frame_files) < self.num_frames:
            frame_files += [frame_files[-1]] * (self.num_frames - len(frame_files))
        else:
            frame_files = frame_files[:self.num_frames]

        frames = []
        for ff in frame_files:
            img = Image.open(ff).convert('RGB')
            if self.transform:
                img = self.transform(img)
            frames.append(img)

        return torch.stack(frames), torch.tensor(label, dtype=torch.long)


transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

full_dataset = DeepfakeDataset(FRAMES_PATH, num_frames=10, transform=transform)

# 80/20 train-val split
train_size = int(0.8 * len(full_dataset))
val_size   = len(full_dataset) - train_size
train_ds, val_ds = torch.utils.data.random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False, num_workers=2)

print(f'✅ Train: {len(train_ds)} | Val: {len(val_ds)}')

## 📌 Phase 6 — Model: EfficientNet (CNN) + LSTM

In [ ]:
class DeepfakeDetector(nn.Module):
    """
    Spatial-Temporal Deepfake Detector
    - EfficientNet-B0: extracts spatial features per frame
    - LSTM: learns temporal patterns across frame sequence
    - FC head: binary classification (real vs fake)
    """
    def __init__(self, num_frames=10, lstm_hidden=256, lstm_layers=2, dropout=0.3):
        super().__init__()

        # CNN: EfficientNet-B0 pretrained on ImageNet
        self.cnn = timm.create_model('efficientnet_b0', pretrained=True, num_classes=0)
        cnn_out_features = self.cnn.num_features  # 1280 for B0

        # LSTM: temporal reasoning across frames
        self.lstm = nn.LSTM(
            input_size=cnn_out_features,
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            dropout=dropout
        )

        # Classifier head
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(lstm_hidden, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 2)  # 2 classes: real, fake
        )

    def forward(self, x):
        # x shape: (batch, num_frames, C, H, W)
        batch, num_frames, C, H, W = x.shape

        # Flatten batch and frames → run all through CNN
        x = x.view(batch * num_frames, C, H, W)
        features = self.cnn(x)  # (batch*frames, 1280)

        # Reshape for LSTM → (batch, frames, features)
        features = features.view(batch, num_frames, -1)

        # LSTM temporal modeling
        lstm_out, _ = self.lstm(features)  # (batch, frames, hidden)
        last_out = lstm_out[:, -1, :]      # Take last timestep

        # Classification
        logits = self.classifier(last_out)
        return logits


model = DeepfakeDetector(num_frames=10, lstm_hidden=256, lstm_layers=2).to(DEVICE)
print('✅ Model ready!')
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

## 📌 Phase 7 — Training

In [ ]:
NUM_EPOCHS = 10
LEARNING_RATE = 1e-4

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)

history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_auc': []}


def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for frames, labels in tqdm(loader, desc='Training'):
        frames, labels = frames.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(frames)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0
    all_preds, all_probs, all_labels = [], [], []

    with torch.no_grad():
        for frames, labels in tqdm(loader, desc='Evaluating'):
            frames, labels = frames.to(DEVICE), labels.to(DEVICE)
            outputs = model(frames)
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            probs = torch.softmax(outputs, dim=1)[:, 1].cpu().numpy()
            preds = outputs.argmax(dim=1).cpu().numpy()
            all_probs.extend(probs)
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    auc = roc_auc_score(all_labels, all_probs)
    return total_loss / len(loader), acc, auc


# Training loop
best_auc = 0
for epoch in range(NUM_EPOCHS):
    print(f'\n🔄 Epoch {epoch+1}/{NUM_EPOCHS}')
    train_loss = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc, val_auc = evaluate(model, val_loader, criterion)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_auc'].append(val_auc)

    print(f'Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | AUC: {val_auc:.4f}')

    # Save best model
    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model.state_dict(), f'{MODELS_PATH}/best_model.pth')
        print(f'💾 Best model saved! AUC: {best_auc:.4f}')

print('\n✅ Training complete!')

## 📌 Phase 8 — Evaluation & Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history['train_loss'], label='Train Loss')
axes[0].plot(history['val_loss'], label='Val Loss')
axes[0].set_title('Loss Curve')
axes[0].legend()

axes[1].plot(history['val_acc'], color='green')
axes[1].set_title('Validation Accuracy')
axes[1].set_ylim(0, 1)

axes[2].plot(history['val_auc'], color='orange')
axes[2].set_title('Validation AUC Score')
axes[2].set_ylim(0, 1)

plt.tight_layout()
plt.savefig(f'{PROJECT_PATH}/training_curves.png')
plt.show()
print(f'\n🏆 Best AUC: {best_auc:.4f}')

## 📌 Phase 9 — Gradio Demo 🚀

In [ ]:
import gradio as gr

# Load best model
model.load_state_dict(torch.load(f'{MODELS_PATH}/best_model.pth', map_location=DEVICE))
model.eval()


def predict_video(video_path):
    """Run inference on an uploaded video."""
    # Extract frames
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    indices = np.linspace(0, total - 1, 10, dtype=int)

    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            img = Image.fromarray(frame_rgb)
            img = transform(img)
            frames.append(img)
    cap.release()

    if len(frames) < 10:
        frames += [frames[-1]] * (10 - len(frames))

    input_tensor = torch.stack(frames).unsqueeze(0).to(DEVICE)  # (1, 10, C, H, W)

    with torch.no_grad():
        output = model(input_tensor)
        probs = torch.softmax(output, dim=1)[0]
        fake_prob = probs[1].item()
        real_prob = probs[0].item()

    if fake_prob > 0.5:
        verdict = f'🚨 FAKE — {fake_prob*100:.1f}% confidence'
    else:
        verdict = f'✅ REAL — {real_prob*100:.1f}% confidence'

    return verdict, {'REAL': round(real_prob, 3), 'FAKE': round(fake_prob, 3)}


demo = gr.Interface(
    fn=predict_video,
    inputs=gr.Video(label='Upload a video'),
    outputs=[
        gr.Textbox(label='Verdict'),
        gr.Label(label='Confidence Scores')
    ],
    title='🎭 Deepfake Detector',
    description='Upload a video to detect whether it is REAL or FAKE using CNN + LSTM spatial-temporal analysis.',
    examples=[]
)

demo.launch(share=True)  # share=True gives a public link for demo!